## Buffer Memory 
* ConversationBufferMemory keeps a list of chat messages in a buffer and passes those into the prompt template

In [2]:
import os 
from dotenv import load_dotenv,find_dotenv
_= load_dotenv(find_dotenv())
groq_api_key = os.environ['GROQ_API_KEY']

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(model='llama-3.3-70b-versatile')

In [19]:
from langchain_core.prompts import(
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([

    ("system","You are a nice chatbot having a conversation with human"),
    MessagesPlaceholder(variable_name="history"),
    ("human","{input}")
])


chain = prompt | llm | StrOutputParser()

history = []

def chat(question):
    global history

    response = chain.invoke({
        "input": question,
        "history" : history
    })

    history.append(("human",question))
    history.append(("ai",response))

    return response


In [ ]:
chat('what is my name')
 

## Conversation window buffer memory 
* similar to the previous one but you can limit the number of conversational exchange stored in memory 
for example you can set it so it only remembers the last 3 questions and answers of the conversation 

In [ ]:
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
# session storage
store = {}

def get_session_history(session_id: str):
    #Create the base history

    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    #Wrap it with ConversationBufferWindowMemory
    memory = ConversationBufferWindowMemory(

        chat_memory= store[session_id], #the underlaying storage
        k=3, # keep only last 3 conversations
        return_messages=True #return as message objects
    )

    #extract the windowed messages

    key = memory.memory_variables[0]
    messages = memory.load_memory_variables({})[key]

    #return a new history with only windowed messages
    store[session_id] = InMemoryChatMessageHistory(messages=messages)
    return store[session_id]

prompt = ChatPromptTemplate(

    messages=[

        ("system","you are the helpfull chatbot"),
        MessagesPlaceholder(variable_name='history'),
        ("human","{input}")

    ]
)

conversation = RunnableWithMessageHistory(

    prompt | llm | StrOutputParser(),
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

def chat(question, session_id = "default"):
    return conversation.invoke(
        {"input":question},
        config={"configurable": {"session_id": session_id}}
    )


In [ ]:
print(chat('hii there',"user1"))